# Divya — RLCD fine-tune of System-1 (Laya) on Indian corporate filings

Kaggle GPU **T4 x2** — the exact hardware Laya's own reference recipe uses. All heavy
compute happens here; the local machine only writes code and scrapes small JSON.

**What this decides.** `docs/loop/STOP_RULE.md` pre-commits, before any training:

> If a fine-tuned Laya does not reach **≥ 0.65 macro-F1 on the 120 held-out NSE
> announcements**, the typed-decision approach is refuted for this domain.

The 120 are excluded from the corpus **by id** and evaluated exactly once, at the end.

**Baselines:** zero-shot macro-F1 **0.147** / accuracy **0.513** on dev; **0.433** accuracy
on the held-out 120 through the Router. Four failed question-set experiments (D-024 → D-026)
established the limit is the model, not the prompt. This tests that conclusion.

In [ ]:
import os, sys, json, time, glob, shutil, zipfile

!pip install -q laya transformers safetensors huggingface_hub 2>&1 | tail -1

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  gpu{i}: {torch.cuda.get_device_name(i)} '
              f'{torch.cuda.get_device_properties(i).total_memory/2**30:.1f} GB')
else:
    print('  NO GPU -- this kernel must run with GPU T4 x2 enabled')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE =', DEVICE)

In [ ]:
def find_dir(filename):
    """Resolve a file's directory by CONTENT, not by path pattern.

    Kaggle creates several directories under /kaggle/input -- including a bare 'datasets'
    owner folder -- and a glob for the dataset name matched the wrong one. Asking which
    directory actually holds the corpus cannot be wrong.
    """
    roots = [d for d in glob.glob('/kaggle/input/*') if os.path.isdir(d)]
    roots += [os.getcwd(), '/kaggle/working']
    for r in roots:
        if os.path.isfile(os.path.join(r, filename)):
            return r
    for r in roots:
        for dirpath, _dirs, files in os.walk(r):
            if filename in files:
                return dirpath
    raise SystemExit(f'could not locate {filename} under {roots}')

DATASET_DIR = find_dir('finetune_corpus.jsonl')
print('dataset dir:', DATASET_DIR)
print('contents:', sorted(os.listdir(DATASET_DIR))[:10])

if os.path.isfile(os.path.join(DATASET_DIR, 'finetune_laya.py')):
    shutil.copy(os.path.join(DATASET_DIR, 'finetune_laya.py'), '/kaggle/working/')
print('trainer in /kaggle/working:', os.path.isfile('/kaggle/working/finetune_laya.py'))

## Zero-shot baseline on dev, for the record

In [ ]:
CORPUS = os.path.join(DATASET_DIR, 'finetune_corpus.jsonl')
HELDOUT = os.path.join(DATASET_DIR, 'nse_v5_eval_set.jsonl')
TRAINER = '/kaggle/working/finetune_laya.py'

base = (f'python {TRAINER} --eval-only --limit-dev 200 --device {DEVICE} '
        f'--corpus {CORPUS} --heldout {HELDOUT}')
print(base)
!{base}


## The fine-tune

Single process with gradient accumulation rather than the reference's DDP. Two T4s are two
cards; DDP buys throughput, not quality. Micro-batch 8 x accum 8 gives an effective batch
of **64**, matching the reference.

In [ ]:
train_cmd = (
    f'python {TRAINER} --epochs 4 --micro-batch 8 --accum 8 --group-size 4 '
    f'--device {DEVICE} --corpus {CORPUS} --heldout {HELDOUT} '
    f'--limit-dev 200 --out /kaggle/working/finetuned'
)
print(train_cmd)
t0 = time.time()
!{train_cmd}
print(f'\nTRAINING WALL TIME: {time.time()-t0:.0f}s')
print('saved:', os.path.isfile('/kaggle/working/finetuned/model.safetensors'))

## Evaluate ONCE on the 120 held-out announcements

This is the number that decides the project. Computed once. If it misses the bar the stop
rule fires and this set is not tuned against again.

In [ ]:
from safetensors.torch import load_file
from transformers import AutoTokenizer
from laya.common import build_model, QTYPES, build_sequence, render_options
from collections import Counter
OUT = '/kaggle/working/finetuned'
if not os.path.isfile(os.path.join(OUT, 'model.safetensors')):
    raise SystemExit('no fine-tuned weights -- the training cell did not save')
cfg = json.load(open(os.path.join(OUT, 'rl_agent_config.json')))
tok = AutoTokenizer.from_pretrained(os.path.join(OUT, 'tokenizer'))
model = build_model(cfg, encoder_dir=os.path.join(OUT, 'encoder'))
model.load_state_dict(load_file(os.path.join(OUT, 'model.safetensors')), strict=True)
model.to(DEVICE).eval()
rows = [json.loads(l) for l in open(HELDOUT) if l.strip()]
options = sorted({r['labels']['event_type'] for r in rows})
crit = {o: '' for o in options}
q = {'t': 'choice', 'ins': 'Classify the corporate event this filing discloses.', 'crit': crit}
k = len(render_options(q))
print('held-out n =', len(rows), '| classes =', options)
tot, ok, pred = Counter(), Counter(), Counter()
with torch.no_grad():
    for r in rows:
        seq, markers = build_sequence(tok, r['content'], q, cfg['max_len'], cfg['head_max_len'])
        if len(markers) != k:
            continue
        ids = torch.tensor([seq], device=DEVICE); att = torch.ones_like(ids)
        mpos = torch.tensor([markers], device=DEVICE)
        mmask = torch.ones_like(mpos, dtype=torch.bool)
        qt = torch.tensor([QTYPES['choice']], device=DEVICE)
        logits, _ = model(ids, att, mpos, mmask, qt)
        pred = options[int(logits.float().reshape(-1).argmax())]
        gold = r['labels']['event_type']
        tot[gold] += 1; pred[p] += 1; ok[gold] += (pred == gold)
n = max(sum(tot.values()), 1)
f1s = []
for lb in options:
    tp = ok[lb]; fn = tot[lb] - tp; fp = pred[lb] - tp   # NOT (N - tot[lb]) - tp
    p = tp / (tp + fp) if tp + fp else 0.0
    rc = tp / (tp + fn) if tp + fn else 0.0
    f1s.append(2 * p * rc / (p + rc) if p + rc else 0.0)
acc = sum(ok.values()) / n
macro_f1 = sum(f1s) / len(f1s) if f1s else 0.0
assert all(pred[l] >= ok[l] for l in options), 'negative false positive -- metric bug'
print(f'HELD-OUT  n={n}  accuracy={acc:.4f}  macro_f1={macro_f1:.4f}')
for lb in options:
    print(f'  {lb:<20} n={tot[lb]:<3} acc={ok[lb]/tot[lb]:.3f}')
BAR = 0.65
verdict = 'PASSED' if macro_f1 >= BAR else 'FIRED'
print(f'\nSTOP RULE (>= {BAR} macro-F1 on held-out): {verdict}')
json.dump({'n': n, 'accuracy': acc, 'macro_f1': macro_f1, 'bar': BAR,
           'verdict': verdict, 'device': DEVICE,
           'per_class': {lb: round(ok[lb] / tot[lb], 4) for lb in options}},
          open('/kaggle/working/HELD_OUT_RESULT.json', 'w'), indent=2)


In [ ]:
import hashlib, zipfile, io
from collections import Counter
from safetensors.torch import load_file

OUT = '/kaggle/working/finetuned'
HELDOUT_IDS = {json.loads(l)['id'] for l in open(HELDOUT) if l.strip()}

# --- the confusion matrix, so macro-F1 is exact next time and the metric bug cannot recur
import torch
from laya.agent import _load_tokenizer
from laya.common import QTYPES, build_sequence, render_options
cfg = json.load(open(os.path.join(OUT, 'rl_agent_config.json')))
tok = _load_tokenizer(os.path.join(OUT, 'tokenizer'), cfg)
model = build_model(cfg, encoder_dir=os.path.join(OUT, 'encoder'))
model.load_state_dict(load_file(os.path.join(OUT, 'model.safetensors')), strict=True)
model.to(DEVICE).eval()
rows = [json.loads(l) for l in open(HELDOUT) if l.strip()]
options = sorted({r['labels']['event_type'] for r in rows})
q = {'t': 'choice', 'ins': 'Classify the corporate event this filing discloses.',
     'crit': {o: '' for o in options}}
k = len(render_options(q))
conf = Counter()   # (gold, pred) -> count. This is what macro-F1 actually needs.
with torch.no_grad():
    for r in rows:
        seq, markers = build_sequence(tok, r['content'], q, cfg['max_len'], cfg['head_max_len'])
        if len(markers) != k: continue
        ids = torch.tensor([seq], device=DEVICE); att = torch.ones_like(ids)
        mpos = torch.tensor([markers], device=DEVICE)
        mmask = torch.ones_like(mpos, dtype=torch.bool)
        qt = torch.tensor([QTYPES['choice']], device=DEVICE)
        lg, _ = model(ids, att, mpos, mmask, qt)
        conf[(r['labels']['event_type'], options[int(lg.float().reshape(-1).argmax())])] += 1
tot = Counter(); pred = Counter(); ok = Counter()
for (g, p), c in conf.items():
    tot[g] += c; pred[p] += c; ok[g] += c if g == p else 0
f1s = []
for lb in options:
    tp = ok[lb]; fn = tot[lb]-tp; fp = pred[lb]-tp
    assert fp >= 0, 'negative false positive'
    pr = tp/(tp+fp) if tp+fp else 0.0; rc = tp/(tp+fn) if tp+fn else 0.0
    f1s.append(2*pr*rc/(pr+rc) if pr+rc else 0.0)
n = max(sum(tot.values()), 1)
res = {'n': n, 'accuracy': round(sum(ok.values())/n, 4),
       'macro_f1': round(sum(f1s)/len(f1s), 4), 'bar': 0.65,
       'confusion': {f'{g}->{p}': c for (g, p), c in sorted(conf.items())},
       'per_class': {lb: round(ok[lb]/tot[lb], 4) for lb in options if tot[lb]}}
res['verdict'] = 'PASSED' if res['macro_f1'] >= 0.65 else 'FIRED'
print('HELD-OUT', json.dumps({k: v for k, v in res.items() if k != 'confusion'}))
print('CONFUSION', json.dumps(res['confusion']))
json.dump(res, open('/kaggle/working/HELD_OUT_RESULT.json', 'w'), indent=2)

# --- the delta: small enough to survive the output API
base_dir = '/kaggle/working/base'
if not os.path.isdir(base_dir):
    os.makedirs(base_dir, exist_ok=True)
    from huggingface_hub import snapshot_download
    p = snapshot_download('convaiinnovations/laya',
                          allow_patterns=['rl_agent_config.json','model.safetensors',
                                          'tokenizer/*','encoder/*'])
    for n in ('rl_agent_config.json','model.safetensors'):
        shutil.copy(os.path.join(p, n), base_dir)
    for sub in ('tokenizer','encoder'):
        shutil.copytree(os.path.join(p, sub), os.path.join(base_dir, sub))

!python export_delta.py --base {base_dir} --tuned {OUT} --out /kaggle/working/delta --keep 0.03
print('FILES:')
for root,_d,fs in os.walk('/kaggle/working/delta'):
    for f in fs: print('  ', os.path.join(root, f), os.path.getsize(os.path.join(root, f)))

# --- delete the full model before the kernel ends.
#
# `kaggle kernels output` streams the WHOLE working directory, and the working directory
# contains a 1.7 GB safetensors. That is why the download timed out and why the weights
# arrived as 0 bytes: the transfer was never small. Only the delta and the results need to
# leave this kernel, and the delta is reconstructible against the stock checkpoint.
shutil.rmtree(OUT, ignore_errors=True)
for f in os.listdir('/kaggle/working'):
    p = os.path.join('/kaggle/working', f)
    print('OUTPUT', f, os.path.getsize(p) if os.path.isfile(p) else '<dir>')
shutil.rmtree(OUT, ignore_errors=True)
shutil.rmtree(base_dir, ignore_errors=True)   # the stock model is NOT shipped; it is on HF
print('OUTPUT CONTENTS:')
for f in sorted(os.listdir('/kaggle/working')):
    p = os.path.join('/kaggle/working', f)
    if os.path.isfile(p): print('  ', f, os.path.getsize(p))
